# 📚 EduDocs AI

Agente conversacional basado en RAG utilizando:

- Google Gemini
- LangChain
- ChromaDB
- Google Colab

Proyecto desarrollado para el Challenge Alura ONE.

---

# 🎯 Objetivo

Desarrollar un agente de inteligencia artificial capaz de responder preguntas
sobre documentos de una plataforma educativa utilizando la arquitectura RAG
(Retrieval-Augmented Generation).

---

# 🏗️ Arquitectura

```text
Documentos
    │
    ▼
PyPDFLoader
    │
    ▼
Document
    │
    ▼
RecursiveCharacterTextSplitter
    │
    ▼
Chunks
    │
    ▼
Google Embeddings
    │
    ▼
ChromaDB
    │
    ▼
Retriever
    │
    ▼
Gemini
    │
    ▼
Respuesta
```

# ==========================================
# Colecta y organización
# ==========================================

En esta etapa se recopilaron y organizaron los documentos que formarán la base de conocimiento del agente.

Se creó una carpeta llamada **documents**, donde se almacenan archivos PDF, Word, Excel, PowerPoint, Markdown, CSV, JSON y HTML pertenecientes a una plataforma educativa.

# ==========================================
# Procesamiento y extracción
# ==========================================

En esta etapa los documentos son cargados, convertidos en objetos Document y posteriormente divididos en pequeños fragmentos (chunks) para facilitar la búsqueda semántica.

In [1]:
# =====================================
# Instalación de dependencias
# =====================================
# Instala todas las librerías necesarias para:
# - Construir el pipeline RAG con LangChain.
# - Procesar documentos de diferentes formatos.
# - Generar embeddings con Gemini.
# - Almacenar vectores en ChromaDB.
# - Leer archivos PDF, Word, Excel, HTML y PowerPoint.

!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-google-genai \
chromadb \
pypdf \
unstructured \
python-docx \
docx2txt \
openpyxl \
beautifulsoup4 \
python-pptx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   

In [2]:
# =====================================
# Importación de librerías
# =====================================

# Librerías estándar de Python
import json
from pathlib import Path

# Librerías para procesar documentos
from openpyxl import load_workbook      # Excel (.xlsx)
from bs4 import BeautifulSoup           # HTML
from pptx import Presentation           # PowerPoint (.pptx)

# Permite acceder de forma segura a la API Key almacenada en Google Colab
from google.colab import userdata

# Clase base utilizada por LangChain para representar documentos
from langchain_core.documents import Document

# Loaders de LangChain para cargar diferentes formatos de archivos
from langchain_community.document_loaders import (
    PyPDFLoader,        # PDF
    Docx2txtLoader,     # Word (.docx)
    TextLoader,         # Texto plano y Markdown
    CSVLoader,          # Archivos CSV
)

# Divide documentos grandes en fragmentos (chunks)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Modelo utilizado para convertir texto en embeddings mediante Google Gemini
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Base de datos vectorial utilizada para almacenar y buscar embeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_26443/1044755086.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


# =====================================
# Función para cargar documentos
# =====================================

In [3]:
# =====================================
# Función para cargar documentos
# =====================================

def cargar_documentos(carpeta):

    # Lista donde se almacenarán todos los documentos cargados
    documents = []

    # Convierte la ruta en un objeto Path para recorrer la carpeta
    carpeta = Path(carpeta)

    # Recorre todos los archivos de la carpeta
    for archivo in carpeta.iterdir():

        try:

            # Obtiene la extensión del archivo en minúsculas
            extension = archivo.suffix.lower()

            # ---------- PDF ----------
            if extension == ".pdf":

                loader = PyPDFLoader(str(archivo))

                documents.extend(loader.load())

                print(f"✔ PDF cargado: {archivo.name}")

            # ---------- WORD ----------
            elif extension == ".docx":

                loader = Docx2txtLoader(str(archivo))

                documents.extend(loader.load())

                print(f"✔ Word cargado: {archivo.name}")

            # ---------- MARKDOWN ----------
            elif extension == ".md":

                loader = TextLoader(
                    str(archivo),
                    encoding="utf-8"
                )

                documents.extend(loader.load())

                print(f"✔ Markdown cargado: {archivo.name}")

            # ---------- CSV ----------
            elif extension == ".csv":

                loader = CSVLoader(str(archivo))

                documents.extend(loader.load())

                print(f"✔ CSV cargado: {archivo.name}")

            # ---------- JSON ----------
            elif extension == ".json":

                # Lee el contenido del archivo JSON
                with open(archivo, "r", encoding="utf-8") as f:
                    contenido = json.load(f)

                # Convierte el JSON en texto legible
                texto = json.dumps(
                    contenido,
                    indent=2,
                    ensure_ascii=False
                )

                # Crea manualmente un objeto Document
                documents.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": str(archivo),
                            "tipo": "json"
                        }
                    )
                )

                print(f"✔ JSON cargado: {archivo.name}")

            # ---------- HTML ----------
            elif extension == ".html":

                # Lee el contenido HTML
                with open(archivo, "r", encoding="utf-8") as f:
                    html = f.read()

                # Elimina las etiquetas HTML y conserva únicamente el texto
                soup = BeautifulSoup(html, "html.parser")

                texto = soup.get_text(
                    separator="\n",
                    strip=True
                )

                # Crea un Document con el texto extraído
                documents.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": str(archivo),
                            "tipo": "html"
                        }
                    )
                )

                print(f"✔ HTML cargado: {archivo.name}")

            # ---------- POWERPOINT ----------
            elif extension == ".pptx":

                # Abre la presentación
                presentacion = Presentation(str(archivo))

                # Recorre todas las diapositivas
                for numero_slide, slide in enumerate(presentacion.slides, start=1):

                    texto = ""

                    # Extrae el texto de cada elemento de la diapositiva
                    for shape in slide.shapes:

                        if hasattr(shape, "text"):
                            texto += shape.text + "\n"

                    texto = texto.strip()

                    # Crea un Document únicamente si la diapositiva contiene texto
                    if texto:

                        documents.append(
                            Document(
                                page_content=texto,
                                metadata={
                                    "source": str(archivo),
                                    "tipo": "powerpoint",
                                    "slide": numero_slide
                                }
                            )
                        )

                print(f"✔ PowerPoint cargado: {archivo.name}")

            # ---------- EXCEL ----------
            elif extension == ".xlsx":

                # Abre el archivo Excel
                workbook = load_workbook(archivo)

                texto = ""

                # Recorre todas las hojas del libro
                for hoja in workbook.worksheets:

                    texto += f"\nHoja: {hoja.title}\n"

                    # Convierte cada fila en texto
                    for fila in hoja.iter_rows(values_only=True):

                        fila_texto = " | ".join(
                            str(celda)
                            for celda in fila
                            if celda is not None
                        )

                        texto += fila_texto + "\n"

                # Crea un Document con todo el contenido del Excel
                documents.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": str(archivo),
                            "tipo": "excel"
                        }
                    )
                )

                print(f"✔ Excel cargado: {archivo.name}")

            # ---------- OTROS ----------
            else:

                print(f"⏭ Archivo no soportado: {archivo.name}")

        # Captura cualquier error sin detener la carga del resto de documentos
        except Exception as e:

            print(f"❌ Error cargando {archivo.name}: {e}")

    # Muestra la cantidad total de documentos cargados
    print(f"\nTotal de documentos cargados: {len(documents)}")

    return documents

In [6]:
# Carga todos los documentos de la carpeta y los convierte en objetos Document
documents = cargar_documentos("documents")

✔ PDF cargado: Reglamento_del_Estudiante_6_paginas.pdf
✔ HTML cargado: Guia_de_Uso_de_la_Plataforma.html
✔ Excel cargado: Programa_de_Becas_y_Afiliados.xlsx
✔ Word cargado: Politica_de_Reembolso_de_Matriculas.docx
✔ Markdown cargado: Preguntas_Frecuentes_Cursos_y_Certificados.md

Total de documentos cargados: 11


In [7]:
# =====================================
# Verificar la cantidad de documentos cargados
# =====================================

len(documents)

11

In [8]:
# =====================================
# Visualizar el primer documento cargado
# =====================================

documents[0]

Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-02T06:27:35+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-02T06:27:35+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'documents/Reglamento_del_Estudiante_6_paginas.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='Reglamento del Estudiante - Plataforma Educativa\n1. Objetivo del reglamento\nDefine las normas generales para garantizar un ambiente de aprendizaje seguro, organizado,\ninclusivo y de calidad para todos los estudiantes. Esta disposición busca establecer procedimientos\nclaros, promover buenas prácticas académicas, facilitar la convivencia entre estudiantes y\ndocentes, proteger los recursos de la plataforma y garantizar una experiencia educativa de calidad.\nSe recomienda revisar periódicamente este reglamento para conocer posibles actualizaciones y\ncumplir con todas 

In [9]:
# =====================================
# Configuración de la API Key
# =====================================

# Obtiene la API Key de Google Gemini almacenada de forma segura en Google Colab
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

In [10]:
# =====================================
# Dividir los documentos en chunks
# =====================================

# Configura el Text Splitter que se encargará de dividir
# los documentos en fragmentos más pequeños
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Tamaño máximo de cada chunk
    chunk_overlap=50     # Superposición entre chunks consecutivos
)

# Divide todos los documentos cargados en chunks
chunks = text_splitter.split_documents(documents)

# ==========================================
# Indexación
# ==========================================

Cada chunk es convertido en un embedding mediante Google Gemini y almacenado dentro de una base de datos vectorial ChromaDB.

In [11]:
# =====================================
# Crear el modelo de embeddings
# =====================================

# Inicializa el modelo de embeddings de Google Gemini, el cual
# convierte texto en vectores numéricos para realizar búsquedas semánticas
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=GOOGLE_API_KEY
)

In [12]:
# =====================================
# Crear la base de datos vectorial
# =====================================

# Crea una base de datos vectorial en ChromaDB a partir de los chunks
# y sus embeddings, permitiendo realizar búsquedas semánticas posteriormente
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2\nPlease retry in 30.623137397s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimen

### Estrategia de Backoff Exponencial para Errores de Cuota

Cuando se trabaja con APIs y se encuentran errores de cuota (`RESOURCE_EXHAUSTED` o `429 Too Many Requests`), una estrategia efectiva es el **backoff exponencial**. Esta técnica consiste en reintentar una solicitud fallida después de un período de espera que aumenta exponencialmente con cada intento consecutivo.

Esto permite que el servicio API se recupere y evita sobrecargar aún más el sistema, lo que podría llevar a un bloqueo permanente o a períodos de espera más largos. Generalmente, se introduce una pequeña aleatoriedad (`jitter`) en el tiempo de espera para evitar que múltiples clientes reintenten al mismo tiempo y causen una nueva oleada de solicitudes.

Aquí tienes una función que implementa esta estrategia y cómo la integrarías en tu código para la creación de la base de datos vectorial.

In [ ]:
import time
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIError

# Definir una función de backoff exponencial
# Reintentará si la excepción es GoogleGenerativeAIError
# Esperará exponencialmente entre 1 y 60 segundos, y reintentará hasta 6 veces
@retry(
    wait=wait_exponential(multiplier=1, min=4, max=60),
    stop=stop_after_attempt(6),
    retry=retry_if_exception_type(GoogleGenerativeAIError)
)
def create_vector_db_with_retries(documents, embeddings):
    print("Intentando crear la base de datos vectorial...")
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings
    )

# Llama a la función con el backoff exponencial
try:
    vector_db = create_vector_db_with_retries(
        documents=chunks,
        embedding=embeddings
    )
    print("Base de datos vectorial creada exitosamente.")
except GoogleGenerativeAIError as e:
    print(f"Error final después de varios reintentos: {e}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")



Este código ahora intentará crear la base de datos vectorial utilizando los chunks y embeddings. Si encuentra un error de `GoogleGenerativeAIError` (como el `RESOURCE_EXHAUSTED` que has estado viendo), esperará una cantidad de tiempo creciente antes de reintentar, hasta un máximo de 6 intentos. Esto debería ayudar a mitigar los problemas de cuota por minuto.

In [ ]:
# =====================================
# Verificar la indexación de los documentos
# =====================================

print(f"Chunks indexados: {vector_db._collection.count()}")

# ==========================================
# Capa de Recuperación (RAG)
# ==========================================

En esta etapa el usuario realiza una pregunta.

LangChain convierte automáticamente esa pregunta en un embedding utilizando el mismo modelo empleado durante la indexación.

Posteriormente, ChromaDB compara ese embedding con todos los embeddings almacenados y recupera los fragmentos (chunks) más similares desde el punto de vista semántico.

Finalmente, estos fragmentos serán utilizados como contexto para que Gemini genere una respuesta en la siguiente etapa.

In [ ]:
# =====================================
# Crear el Retriever
# =====================================

# Crea el componente encargado de recuperar los fragmentos
# más relevantes desde la base de datos vectorial utilizando
# la similitud entre la pregunta y los embeddings almacenados
retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 3   # Número de chunks que se recuperarán en cada búsqueda
    }
)

In [ ]:
# =====================================
# Definir la pregunta del usuario
# =====================================

# Simula la consulta realizada por un usuario al agente RAG
pregunta = "¿Cómo obtengo mi certificado?"

In [ ]:
# =====================================
# Recuperar información relevante
# =====================================

# Envía la pregunta al Retriever para recuperar los documentos
# más relevantes utilizando búsqueda semántica
resultados = retriever.invoke(pregunta)

In [ ]:
# =====================================
# Mostrar los resultados de la búsqueda
# =====================================

# Recorre los documentos recuperados y muestra su contenido
# junto con el archivo del que fueron obtenidos
for i, doc in enumerate(resultados, start=1):

    # Imprime un encabezado para cada resultado
    print("=" * 60)
    print(f"Resultado {i}")
    print("=" * 60)

    # Muestra el contenido del fragmento recuperado
    print(doc.page_content)

    # Muestra el documento de origen del fragmento
    print("\nFuente:", doc.metadata["source"])

    print()

En esta etapa se implementó la recuperación de información (Retrieval).

Se creó un Retriever a partir de la base vectorial ChromaDB, el cual recibe una pregunta del usuario y recupera los fragmentos más relevantes utilizando búsqueda semántica.

Los fragmentos recuperados serán utilizados como contexto por Gemini en la siguiente etapa.

## Nota

En este proyecto no se implementó filtrado por metadatos ni un modelo de reranking, ya que se trata de un MVP con un único documento de una plataforma educativa.

La recuperación se realiza mediante búsqueda semántica utilizando ChromaDB y un Retriever de LangChain, lo cual es suficiente para cumplir

# ==========================================
# Generación de respuestas
# ==========================================

En esta etapa se conecta el Retriever con el modelo Gemini.

Cuando el usuario realiza una pregunta, el Retriever recupera los fragmentos más relevantes de la base vectorial.

Posteriormente, dichos fragmentos son enviados junto con la pregunta a Gemini para generar una respuesta basada únicamente en el contexto recuperado.

Finalmente, la respuesta se devuelve al usuario indicando la fuente de la información utilizada.

In [ ]:
# =====================================
# Importar el modelo conversacional de Gemini
# =====================================

# Importa la clase utilizada para integrar un modelo
# conversacional de Google Gemini dentro de LangChain
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# =====================================
# Configurar el modelo de lenguaje (LLM)
# =====================================

# Configura el modelo de lenguaje de Google Gemini que generará
# las respuestas utilizando el contexto recuperado por el Retriever
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",          # Modelo de lenguaje
    google_api_key=GOOGLE_API_KEY,     # Credenciales de acceso a la API
    temperature=0                      # Reduce la aleatoriedad para obtener respuestas más consistentes
)

# ==========================================
# Construcción del Prompt
# ==========================================

El prompt define las instrucciones que seguirá Gemini para responder.

Se le indica que únicamente utilice el contexto recuperado por el Retriever, evitando generar información que no exista en los documentos.

Además, siempre deberá indicar la fuente utilizada y, si no encuentra la respuesta, informarlo explícitamente.

In [ ]:
# =====================================
# Importar la plantilla de Prompt
# =====================================

# Importa la clase que permite construir el Prompt que recibirá
# el modelo de lenguaje junto con el contexto recuperado
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# =====================================
# Crear el Prompt del agente
# =====================================

# Define las instrucciones que recibirá el modelo de lenguaje.
# El Prompt establece el comportamiento del agente indicando
# cómo debe responder y qué información puede utilizar.
prompt = ChatPromptTemplate.from_template("""
Eres un asistente virtual para una plataforma educativa.

Responde únicamente utilizando la información contenida en el contexto.

Si la respuesta no se encuentra en el contexto responde exactamente:

"No encontré esa información dentro de los documentos disponibles."

Nunca inventes información.

Si la información proviene de uno o varios documentos del contexto,
menciónalos al final de la respuesta cuando sea posible.

Contexto:
{context}

Pregunta:
{question}
""")

In [ ]:
# =====================================
# Importar componentes para la cadena RAG
# =====================================

# Importa las clases que permiten construir el flujo de trabajo
# del agente y procesar la respuesta generada por el modelo
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# =====================================
# Construir la cadena RAG
# =====================================

# Crea el flujo de trabajo del agente RAG. La pregunta del usuario
# se utiliza para recuperar el contexto relevante desde la base de
# datos vectorial, construir el Prompt, enviarlo al modelo Gemini
# y devolver la respuesta como texto.
rag_chain = (
    {
        # Recupera los fragmentos más relevantes desde ChromaDB
        "context": retriever,

        # Pasa la pregunta del usuario al Prompt
        "question": RunnablePassthrough()
    }

    # Construye el Prompt utilizando el contexto recuperado
    | prompt

    # Envía el Prompt al modelo de lenguaje
    | llm

    # Convierte la respuesta del modelo en texto plano
    | StrOutputParser()
)

In [ ]:
# =====================================
# Consultar el agente
# =====================================

# Envía una pregunta al agente RAG y muestra la respuesta generada
respuesta = rag_chain.invoke(
    "¿Cómo obtengo mi certificado?"
)

print(respuesta)